# Single-Qubit T1 (Amplitude Damping)

Derive symbolic transition matrix for amplitude damping in computational basis.

## Setup

In [1]:
import sympy as sp
from sympy import Matrix, sqrt, symbols
import mct
import numpy as np

lam = symbols('lambda', real=True, positive=True)

## Derive Transition Matrix

In [2]:
# Amplitude damping Kraus operators
E0 = Matrix([[1, 0], [0, sqrt(1 - lam)]])
E1 = Matrix([[0, sqrt(lam)], [0, 0]])

# Derive symbolic transition matrix
P, metadata = mct.derive_transition_matrix(
    [E0, E1],
    mct.computational_basis,
    channel_name="T1"
)

# Display with LaTeX formatting
mct.print_matrix_latex(P, name="P(λ)")

$P(λ) = \displaystyle \left[\begin{matrix}1 & \lambda\\0 & \sqrt{1 - \lambda} \overline{\sqrt{1 - \lambda}}\end{matrix}\right]$

## Verify Stochasticity

In [3]:
# Test at specific λ values
for lam_val in [0.0, 0.1, 0.5, 0.9, 1.0]:
    P_num, is_valid = mct.substitute_and_verify(P, lam, lam_val, name=f"λ={lam_val}")
    assert is_valid, f"Failed at λ={lam_val}"

print("✓ All parameter values are stochastic")

λ=0.0
  λ = 0.0
  P_numeric =
[[1. 0.]
 [0. 1.]]
  Column sums: [1. 1.]
  ✓ Valid: True

λ=0.1
  λ = 0.1
  P_numeric =
[[1.  0.1]
 [0.  0.9]]
  Column sums: [1. 1.]
  ✓ Valid: True

λ=0.5
  λ = 0.5
  P_numeric =
[[1.  0.5]
 [0.  0.5]]
  Column sums: [1. 1.]
  ✓ Valid: True

λ=0.9
  λ = 0.9
  P_numeric =
[[1.  0.9]
 [0.  0.1]]
  Column sums: [1. 1.]
  ✓ Valid: True

λ=1.0
  λ = 1.0
  P_numeric =
[[1. 1.]
 [0. 0.]]
  Column sums: [1. 1.]
  ✓ Valid: True

✓ All parameter values are stochastic


## Use as Markov Chain

In [4]:
# Create numeric MarkovChain at λ=0.3
P_numeric = P.subs(lam, 0.3)
P_np = np.array(P_numeric, dtype=float)

mc = mct.MarkovChain(P_np, state_labels=['|0⟩', '|1⟩'], name='T1')
result = mc.validate_stochasticity()

print(f"MarkovChain: {mc.name}")
print(f"Valid: {result['is_valid']}")
print(f"Column sums: {result['column_sums']}")

MarkovChain: T1
Valid: True
Column sums: [1. 1.]


## Generate Sampling Code

In [5]:
# Generate executable Python sampling function
code = mct.markov_chain((P, metadata), parameter_values={'lambda': 0.3})

print(code[:400])  # Show first 400 chars
print(f"\n... ({len(code)} total characters)")

def sample_markov_step(current_state: int) -> int:
    """Sample next state using cumulative probabilities."""
    import random
    r = random.random()

    if current_state == 0:  # 0
        if r < 1.000000000000000:
            return 0  # -> 0
        else:  # cumsum = 1.000000000000000
            return 1  # -> 1

    if current_state == 1:  # 1
        if r < 0.300000000000000:
           

... (491 total characters)
